# TVB Whole-Brain Epilepsy Workflow: Epileptor Stimulus-Evoked Response

This notebook builds a whole-brain simulation of the default 76-region Hagmann connectome using the **Epileptor** neural-mass model with heterogeneous excitability (`x0`). A brief stimulus is delivered to the visual regions V1 and V2 via a `PulseTrain` to study seizure-like propagation dynamics.

---


## Step 1 – Imports and default connectivity

Load the standard TVB library and the default 76-region connectivity. We set the conduction speed to **4.0 mm/ms** and call `configure()` on the connectivity object before use.

In [ ]:
from tvb.simulator.lab import *
from tvb.datatypes.region_mapping import RegionMapping
from tvb.datatypes.surfaces import CorticalSurface
import numpy
import matplotlib.pyplot as plt
from scipy import signal

# Default 76-region Hagmann connectivity
conn = connectivity.Connectivity.from_file()
conn.speed = numpy.array([4.0])
conn.configure()

# Surface and region mapping needed for EEG / iEEG forward models
surf = CorticalSurface.from_file()
surf.configure()

rm = RegionMapping.from_file('regionMapping_16k_76.txt')
rm.connectivity = conn
rm.surface = surf
rm.configure()

print("Number of regions:", conn.number_of_regions)
print("V1 label:", conn.region_labels[35])
print("V2 label:", conn.region_labels[36])

## Step 2 – Configure `Epileptor` with heterogeneous `x0`

The `Epileptor` is a 6-dimensional neural-mass model capturing seizure dynamics via a fast subsystem (`x1`, `y1`, `x2`, `y2`) and a slow permittivity variable (`z`).

We assign a **heterogeneous `x0`** across the 76 regions so that some nodes are closer to the epileptogenic threshold while others remain in a healthy regime. This mimics patient-specific networks where focal regions are more excitable.

| Parameter | Value | Role |
|-----------|-------|------|
| `a`       | 1.0   | coefficient x1 |
| `b`       | 3.0   | coefficient y1 |
| `c`       | 1.0   | coefficient z |
| `d`       | 5.0   | coefficient x2 |
| `r`       | 0.00035 | time-scale separation |
| `s`       | 4.0   | nonlinear gain |
| `Iext`    | 3.1   | external drive to x2 |
| `Iext2`   | 0.45  | external drive to x1 |
| `x0`      | uniform(-2.5, -1.5) | heterogeneous excitability |


In [ ]:
numpy.random.seed(42)
x0_vals = numpy.random.uniform(low=-2.5, high=-1.5, size=(conn.number_of_regions,))

model = models.Epileptor(
    a=numpy.array([1.0]),
    b=numpy.array([3.0]),
    c=numpy.array([1.0]),
    d=numpy.array([5.0]),
    r=numpy.array([0.00035]),
    s=numpy.array([4.0]),
    Iext=numpy.array([3.1]),
    Iext2=numpy.array([0.45]),
    x0=x0_vals,
)

print("State variables:", model.state_variables)
print("Number of state variables:", model.nvar)
print("variables_of_interest:", model.variables_of_interest)
print("x0 range: {: .3f} to {: .3f}".format(model.x0.min(), model.x0.max()))

## Step 3 – Define coupling: `coupling.Difference`

We use `coupling.Difference(a=1.0)` to compute the difference between the source and target node activities weighted by the structural connectivity. This is the standard choice for Epileptor network simulations.

In [ ]:
coup = coupling.Difference(
    a=numpy.array([1.0])
)
print("Coupling function configured:", type(coup).__name__)

## Step 4 – Define V1/V2 stimulus using `PulseTrain`

We construct a region-level stimulus that targets:
- **Region 35** (V1) with amplitude **3.5**
- **Region 36** (V2) with amplitude **1.5**

The temporal profile is an impulsive `PulseTrain` with:
- **onset** = 500 ms
- **tau**   = 5 ms (brief pulse width)
- **T**     = 10 000 ms (repetition interval set >> simulation length so only a single pulse is delivered)

> **Burn-in rule:** Because the stimulus onset is at 500 ms, we do not discard the first 1000 ms as burn-in. Only brief pre-stimulus transients (≤ 100 ms) are excluded, or burn-in is skipped entirely.

In [ ]:
# Stimulus weight vector: shape (n_regions, 1)
stim_weights = numpy.zeros((conn.number_of_regions, 1))
stim_weights[35] = numpy.array([[3.5]])   # V1
stim_weights[36] = numpy.array([[1.5]])   # V2

# Temporal profile: single brief pulse
eqn_t = equations.PulseTrain()
eqn_t.parameters["onset"] = 500.0   # ms
eqn_t.parameters["tau"]   = 5.0     # ms
eqn_t.parameters["T"]     = 10000.0 # ms – set large to avoid repetition

stimulus = patterns.StimuliRegion(
    temporal=eqn_t,
    connectivity=conn,
    weight=stim_weights
)
stimulus.configure()

print("Stimulus configured. Active nodes:", numpy.where(stim_weights.squeeze() > 0)[0])

## Step 5 – Configure `HeunStochastic` integrator with length-6 noise vector

Because `Epileptor` has six state variables (`x1`, `y1`, `z`, `x2`, `y2`, `g`), the noise covariance `nsig` must be a **length-6 vector**. We apply small additive noise to the fast variables and leave the slow variable `z` and coupling variable `g` noise-free. A `dt` of **0.05 ms** (2⁻⁴ ms) provides stable integration.

In [ ]:
nsig = numpy.array([0.01, 0.01, 0.0, 0.01, 0.01, 0.0])
hiss = noise.Additive(nsig=nsig)

heunint = integrators.HeunStochastic(
    dt=0.05,
    noise=hiss
)

print("Integrator:", type(heunint).__name__)
print("dt =", heunint.dt)
print("nsig shape =", heunint.noise.nsig.shape)

## Step 6 – Attach monitors: `TemporalAverage`, `EEG`, and `iEEG`

We attach three monitors:
1. **`monitors.TemporalAverage`** – sampled every 1 ms, captures `x2 - x1` and `z`.
2. **`monitors.EEG`** – requires a surface `region_mapping` and default EEG sensors.
3. **`monitors.iEEG`** – requires the same `region_mapping` and default internal (sEEG) sensors.

> ⚠️ The region mapping is linked to the cortical surface so that forward-projected signals (EEG / iEEG) have a valid spatial source basis even though the dynamics are simulated at the region level.

In [ ]:
mon_tavg = monitors.TemporalAverage(period=1.0)

mon_eeg = monitors.EEG(
    sensors=sensors.SensorsEEG.from_file('eeg_unitvector_62.txt.bz2'),
    region_mapping=rm,
    period=1.0
)

mon_ieeg = monitors.iEEG(
    sensors=sensors.SensorsInternal.from_file('seeg_39.txt.bz2'),
    region_mapping=rm,
    period=1.0
)

print("Monitors configured:", [type(m).__name__ for m in (mon_tavg, mon_eeg, mon_ieeg)])

## Step 7 – Build and run the `Simulator`

We instantiate `simulator.Simulator` with all components, call `configure()`, and run for **5000 ms** so that the post-stimulus window (500 ms → 5000 ms) is long enough to observe propagation.

In [ ]:
sim = simulator.Simulator(
    model=model,
    connectivity=conn,
    coupling=coup,
    integrator=heunint,
    monitors=(mon_tavg, mon_eeg, mon_ieeg),
    stimulus=stimulus
)
sim.configure()

print("Simulator configured. Starting run ...")

(t_tavg, y_tavg), (t_eeg, y_eeg), (t_ieeg, y_ieeg) = sim.run(simulation_length=5000.0)

print("Done.")
print("TemporalAverage shape:", y_tavg.shape, "time range:", t_tavg.min(), "-", t_tavg.max())
print("EEG shape:", y_eeg.shape)
print("iEEG shape:", y_ieeg.shape)

## Step 8 – Post-hoc analysis

We carry out three complementary analyses on the **TemporalAverage** output (region-level `x2 - x1` and `z`).

### 8.1 Align analysis windows with the scientific question
Because the stimulus arrives at 500 ms, the functional connectivity (FC) and spectral analyses are computed on a post-stimulus window **\[600 ms, 5000 ms\]** rather than the entire trace. This avoids conflating pre-stimulus resting dynamics with the evoked propagation epoch.

In [ ]:
# y_tavg shape: (time, variables_of_interest, nodes, modes)
# variable 0 = x2 - x1 (fast population activity proxy)
# variable 1 = z (slow permittivity)
var_fast = y_tavg[:, 0, :, 0]   # (time, nodes)
var_slow = y_tavg[:, 1, :, 0]   # (time, nodes)

# Post-stimulus mask (600 ms -> end)
post_mask = t_tavg >= 600.0
t_post = t_tavg[post_mask]
fast_post = var_fast[post_mask, :]
slow_post = var_slow[post_mask, :]

print("Post-stimulus samples:", fast_post.shape[0])

### 8.2 Functional connectivity (Pearson correlation)

In [ ]:
fc_fast = numpy.corrcoef(fast_post.T)
print("FC matrix shape:", fc_fast.shape)

### 8.3 Power spectral density (Welch) for V1 and V2

In [ ]:
fs = 1000.0  # monitors sampled at 1 ms => 1 kHz

# V1 = node 35, V2 = node 36
f_v1, Pxx_v1 = signal.welch(fast_post[:, 35], fs=fs, nperseg=256)
f_v2, Pxx_v2 = signal.welch(fast_post[:, 36], fs=fs, nperseg=256)

## Step 9 – Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Panel A: Time series of x2-x1 for selected regions ---
ax = axes[0, 0]
for idx, label in [(35, "rV1"), (36, "rV2"), (40, "lAMYG"), (0, "rA1"), (70, "lTCPOL")]:
    ax.plot(t_tavg, var_fast[:, idx], label=label, alpha=0.8)
ax.axvline(500.0, color='red', linestyle='--', label='stimulus onset')
ax.set_xlabel("Time (ms)")
ax.set_ylabel("x2 - x1")
ax.set_title("Epileptor fast variable (x2 - x1)")
ax.legend(loc='upper right')
ax.set_xlim(0, 5000)

# --- Panel B: Slow variable z for same regions ---
ax = axes[0, 1]
for idx, label in [(35, "rV1"), (36, "rV2"), (40, "lAMYG"), (0, "rA1"), (70, "lTCPOL")]:
    ax.plot(t_tavg, var_slow[:, idx], label=label, alpha=0.8)
ax.axvline(500.0, color='red', linestyle='--', label='stimulus onset')
ax.set_xlabel("Time (ms)")
ax.set_ylabel("z")
ax.set_title("Epileptor slow permittivity (z)")
ax.legend(loc='upper right')
ax.set_xlim(0, 5000)

# --- Panel C: Post-stimulus functional connectivity ---
ax = axes[1, 0]
im = ax.imshow(fc_fast, cmap='viridis', vmin=-1, vmax=1)
ax.set_title("Post-stimulus FC (fast variable)\n600 ms → 5000 ms")
ax.set_xlabel("Region")
ax.set_ylabel("Region")
plt.colorbar(im, ax=ax, shrink=0.8)

# --- Panel D: PSD for V1 and V2 ---
ax = axes[1, 1]
ax.semilogy(f_v1, Pxx_v1, label='V1')
ax.semilogy(f_v2, Pxx_v2, label='V2', alpha=0.7)
ax.set_xlim(0, 50)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD (a.u.)")
ax.set_title("Power spectral density (post-stimulus)")
ax.legend()

plt.tight_layout()
plt.show()

## Step 10 – Summary statistics

Compute simple summary statistics to verify the output scales.

In [ ]:
print("Mean FC (fast variable):", numpy.mean(numpy.abs(fc_fast[numpy.triu_indices_from(fc_fast, k=1)])))
print("Max post-stimulus x2-x1 (V1):", fast_post[:, 35].max())
print("Max post-stimulus x2-x1 (V2):", fast_post[:, 36].max())
print("Mean z (V1) post-stimulus:", slow_post[:, 35].mean())
print("Peak PSD frequency V1:", f_v1[numpy.argmax(Pxx_v1)], "Hz")